In [ ]:
#python -m pip install --upgrade pip
#pip install transformers==4.46.3 accelerate bitsandbytes peft trl datasets huggingface_hub wandb scipy

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer,BitsAndBytesConfig
model_id = 'Qwen/Qwen2.5-0.5B-Instruct'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16    
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map='auto')

In [ ]:
# 데이터셋을 특정 모델용으로 변환
def format_prompt(example):
    messages = [
        {'role':'system','content':'You are a helpful AI assistant, Respond in Korean.'},
        {'role':'user','content':example['instruction'}
    ]
    text = tokenizer.apply_chat_template(messages, tokenizer=False, add_generation_prompt=True)
    text += example['output']+ tokenizer.eos_token
    return {'text':text}